<a href="https://colab.research.google.com/github/raghavarajunithisha-lab/humangate_demo/blob/main/applied_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install typesafe-sdk

In [10]:
"""
Governance Gate Demo — Jev (TypeSafe) applied to Opus-style regulated workflows
=================================================================================
Purpose: a small, honest test of whether TypeSafe's Jev — a System One model
that returns typed, structured decisions instead of generated text — can
replace a full LLM call for the "does this need a human?" gate that sits
inside Opus's Run/Optimise loop for regulated enterprises (healthcare,
insurance, finance).

Built directly against TypeSafe's documented Python SDK and API contract:
  - Introduction:  https://docs.typesafe.ai/introduction
  - Quick Start:   https://docs.typesafe.ai/introduction/quickstart
  - Choice primitive: https://docs.typesafe.ai/primitives/choice

Per the docs, each question is one of three primitives (Choice, Score, Noul).
This demo uses two Choice questions per case — routing and risk level — each
evaluated in parallel against the same "state" (the case description) in a
single request, as the docs describe.

Getting an API key:
    Just sign in at https://typesafe.ai/ and create an API key.

Setup (Google Colab):
    1. Click the key icon in the left sidebar -> Add new secret
    2. Name it TYPESAFE_API_KEY, paste the API key you created at
       https://typesafe.ai/ as the value, and enable notebook access
    3. Run this cell

Setup (local):
    pip install typesafe-sdk
    export TYPESAFE_API_KEY="your-api-key-from-typesafe.ai"
    python opus_governance_gate_demo.py
"""

import os
import time
import csv

from typesafe_sdk import Choice, TypeSafeClient

# --- Colab secret support -----------------------------------------------
# If running in Colab, pull the key from the secrets vault and expose it as
# an environment variable, since TypeSafeClient() reads TYPESAFE_API_KEY
# from the environment automatically (per the Quick Start's "Code it" section).
try:
    from google.colab import userdata
    os.environ.setdefault("TYPESAFE_API_KEY", userdata.get("TYPESAFE_API_KEY") or "")
except ImportError:
    pass  # not running in Colab; assume TYPESAFE_API_KEY is already set

MODEL = "jev-latest"

# Input price only — per TypeSafe's Sep 2026 announcement, output tokens are
# free ("too cheap to meter"). Verify against your dashboard before quoting
# this externally, since published rates can change.
INPUT_PRICE_PER_MILLION = 0.042

# ---------------------------------------------------------------------------
# Scenarios modeled directly on Opus's own stated domains: regulated
# healthcare/insurance/finance workflows, and the human-approval gates that
# sit inside their "Run" and "Optimise" stages.
# ---------------------------------------------------------------------------
SCENARIOS = [
    "Insurance claim requests $4,200 reimbursement for an out-of-network procedure not on the pre-approved list.",
    "Patient intake form flags a known drug allergy that conflicts with the newly prescribed medication.",
    "Routine annual eye exam claim for $85, fully within the standard covered-procedures list.",
    "Vendor invoice for $312 matches purchase order exactly and vendor is on the approved supplier list.",
    "Vendor invoice for $58,000 has no matching purchase order on file.",
    "Employee expense report includes a $1,900 client dinner with no itemized receipt attached.",
    "Loan application shows income verification document dated 14 months ago; policy requires within 12 months.",
    "Refund request for a $22 subscription overcharge, matching billing system's own error log.",
    "New vendor onboarding request from a company incorporated in a sanctioned jurisdiction.",
    "Standard prescription refill request with no dosage change and prior-authorization already on file.",
    "Insurance claim for $190,000 hospital stay flagged by system as statistically unusual for the diagnosis code.",
    "Internal workflow change request to auto-approve claims under $50 with no human review.",
    "Contractor payment of $4,500 matches signed contract terms exactly, contractor previously verified.",
    "Data access request from a new hire for a regulated patient records database, manager approval pending.",
    "Routine PTO request with no blackout-period conflict and sufficient balance.",
]

# Two Choice primitives (docs.typesafe.ai/primitives/choice), built with the
# SDK's Choice() class rather than a raw dict — this is the documented,
# typed way to construct a question.
QUESTIONS = {
    "routing_decision": Choice(
        instructions="Should this case be auto-approved and proceed automatically, or escalated to a human reviewer?",
        criteria={
            "auto_approve": "Low-risk, policy-compliant, no red flags — safe to process automatically.",
            "escalate_to_human": "Ambiguous, high-value, policy exception, or contains a compliance/risk flag.",
        },
    ),
    "risk_level": Choice(
        instructions="What is the overall risk level of this case for a regulated enterprise workflow?",
        criteria={
            "low": "Minimal risk, routine, well within policy.",
            "medium": "Some ambiguity or minor policy deviation worth noting.",
            "high": "Significant compliance, financial, or safety risk.",
        },
    ),
}


def evaluate_scenario(client: TypeSafeClient, state: str) -> dict:
    """Send one scenario as `state` to Jev via the documented system_one() call."""
    start = time.perf_counter()
    response = client.system_one(state=state, questions=QUESTIONS, model=MODEL)
    latency_ms = (time.perf_counter() - start) * 1000

    routing = response.answers["routing_decision"]
    risk = response.answers["risk_level"]
    return {
        "scenario": state,
        "routing_decision": routing.choice,
        "routing_confidence": round(routing.confidence, 3),
        "risk_level": risk.choice,
        "risk_confidence": round(risk.confidence, 3),
        "latency_ms": round(latency_ms, 1),
        "input_tokens": response.usage.input_tokens,
        "cost_usd": round((response.usage.input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION, 6),
    }


def main():
    if not os.environ.get("TYPESAFE_API_KEY"):
        raise SystemExit("Set TYPESAFE_API_KEY in your environment (or Colab secrets) first.")

    client = TypeSafeClient()  # reads TYPESAFE_API_KEY from the environment, per the docs

    rows = []
    print(f"{'#':<3} {'Routing':<18} {'Conf':<6} {'Risk':<8} {'Conf':<6} {'Latency(ms)':<12} {'Cost($)':<10} Scenario")
    print("-" * 130)

    for i, scenario in enumerate(SCENARIOS, 1):
        try:
            result = evaluate_scenario(client, scenario)
        except Exception as e:
            print(f"{i:<3} ERROR: {e}")
            continue

        rows.append(result)
        print(
            f"{i:<3} {result['routing_decision']:<18} {result['routing_confidence']:<6} "
            f"{result['risk_level']:<8} {result['risk_confidence']:<6} "
            f"{result['latency_ms']:<12.1f} {result['cost_usd']:<10.6f} {scenario[:45]}"
        )

    if not rows:
        raise SystemExit("No scenarios succeeded — check your API key and network access.")

    with open("governance_gate_results.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    total_cost = sum(r["cost_usd"] for r in rows)
    avg_latency = sum(r["latency_ms"] for r in rows) / len(rows)
    avg_confidence = sum(r["routing_confidence"] for r in rows) / len(rows)
    escalated = sum(1 for r in rows if r["routing_decision"] == "escalate_to_human")

    print("\n--- Summary ---")
    print(f"Scenarios run:            {len(rows)}")
    print(f"Escalated to human:       {escalated} / {len(rows)}")
    print(f"Avg routing confidence:   {avg_confidence:.3f}")
    print(f"Avg latency:              {avg_latency:.1f} ms")
    print(f"Total cost:               ${total_cost:.6f}")
    print("Saved: governance_gate_results.csv")


if __name__ == "__main__":
    main()

#   Routing            Conf   Risk     Conf   Latency(ms)  Cost($)    Scenario
----------------------------------------------------------------------------------------------------------------------------------
1   escalate_to_human  1.0    medium   0.75   446.4        0.000020   Insurance claim requests $4,200 reimbursement
2   escalate_to_human  1.0    high     1.0    124.1        0.000020   Patient intake form flags a known drug allerg
3   auto_approve       1.0    low      1.0    159.8        0.000020   Routine annual eye exam claim for $85, fully 
4   auto_approve       0.98   low      1.0    150.1        0.000020   Vendor invoice for $312 matches purchase orde
5   escalate_to_human  1.0    high     0.92   134.3        0.000020   Vendor invoice for $58,000 has no matching pu
6   escalate_to_human  1.0    high     0.28   162.4        0.000020   Employee expense report includes a $1,900 cli
7   escalate_to_human  1.0    medium   0.32   124.7        0.000020   Loan application shows i